In [131]:
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1. Definir la URL de la página web (ajusta la ruta exacta si es otra sección)
url = "https://www.curses.cat/classificacions/"

# Configurar cabeceras para simular un navegador real y evitar bloqueos de seguridad
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    print(f"Conectando a {url}...")
    # 2. Hacer la petición de descarga a internet
    response = requests.get(url, headers=headers)

    # Comprobar que la página ha respondido correctamente (Código 200)
    response.raise_for_status()

    # 3. Parsear el HTML descargado en vivo
    soup = BeautifulSoup(response.text, "html.parser")

    # Buscamos todas las filas (tr) de la tabla
    rows = soup.find_all("tr")

    llista_curses = []

    # 4. Procesar las filas dinámicamente
    for row in rows:
        cols = row.find_all("td")

        # Aseguramos que la fila tenga las celdas de datos estructuradas
        if len(cols) >= 3:
            # Extraer la fecha (dentro del span oculto)
            span_data = cols[0].find("span")
            data_neta = (
                span_data.text.split()[0] if span_data else ""
            )  # Formato YYYY-MM-DD

            # Extraer el nombre de la carrera y su enlace
            a_tag = cols[1].find("a")
            if a_tag:
                url_cursa = a_tag["href"]
                nom_cursa = a_tag.text.strip()
            else:
                url_cursa = ""
                nom_cursa = ""

            # Extraer la población (limpiando el nombre de la carrera del texto global)
            text_cel_la2 = cols[1].get_text(strip=True)
            poblacio = (
                text_cel_la2.replace(nom_cursa, "").strip().strip("-").strip()
            )

            # Extraer el tipo de carrera
            b_tag = cols[2].find("b")
            tipus = b_tag.text.strip() if b_tag else ""

            # Extraer las distancias
            text_cel_la3 = cols[2].get_text(separator="|", strip=True)
            parts_cel_la3 = text_cel_la3.split("|")
            distancies = (
                parts_cel_la3[-1].strip() if len(parts_cel_la3) > 1 else ""
            )

            # Guardar en el diccionario
            llista_curses.append(
                {
                    "Data": data_neta,
                    "Cursa": nom_cursa,
                    "Població": poblacio,
                    "Tipus": tipus,
                    "Distàncies": distancies,
                    "URL": url_cursa,
                }
            )

    # 5. Convertir a un DataFrame de Pandas
    df = pd.DataFrame(llista_curses)

    if not df.empty:
        print(f"\n¡Éxito! Se han extraído {len(df)} carreras directamente de la web.")

        # Guardar directamente el resultado en un archivo Excel
        df.to_excel("../../data/raw/cursescat/Curses_CAT.xlsx", index=False)
        print("Datos guardados correctamente en 'curses_catalunya_vivo.xlsx'")

        # Mostrar una pequeña vista previa en la consola
        print("\nVista previa de los datos extraídos:")
        print(df.head(5))
    else:
        print(
            "No se encontraron datos en la tabla. Verifica si la estructura HTML de la página ha cambiado."
        )

except requests.exceptions.RequestException as e:
    print(f"\nError de conexión al intentar acceder a la web: {e}")

Conectando a https://www.curses.cat/classificacions/...

¡Éxito! Se han extraído 283 carreras directamente de la web.
Datos guardados correctamente en 'curses_catalunya_vivo.xlsx'

Vista previa de los datos extraídos:
         Data                                 Cursa  \
0  2026-06-21                    Olla de Tapis 2026   
1  2026-06-07                        La Cirera 2026   
2  2026-06-07                     Espardenyada 2026   
3  2026-05-31  3ª Edició d'els Corriols de Colomers   
4  2026-05-31            11ª Edició Marxa de l'Ordi   

                      Població              Tipus  Distàncies  \
0  Tapis - Maçanet de Cabrenys              Trail  9km i 21km   
1                     Terrades  Marxa Cursa Trail  7km i 13km   
2        Sant Jaume de Llierca        Marxa trail  9km i 16km   
3                     Colomers        Marxa cursa  7km i 14km   
4                        Ordis        Marxa Cursa  5km i 10km   

                                                 URL  
0  ht

In [132]:
df_curses = pd.read_excel("../../data/raw/cursescat/Curses_CAT.xlsx")
df_curses

,Data,Cursa,Població,Tipus,Distàncies,URL
0,2026-06-21,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,Trail,9km i 21km,https://www.curses.cat/classificacions/classif...
1,2026-06-07,La Cirera 2026,Terrades,Marxa Cursa Trail,7km i 13km,https://www.curses.cat/classificacions/classif...
2,2026-06-07,Espardenyada 2026,Sant Jaume de Llierca,Marxa trail,9km i 16km,https://www.curses.cat/classificacions/classif...
3,2026-05-31,3ª Edició d'els Corriols de Colomers,Colomers,Marxa cursa,7km i 14km,https://www.curses.cat/classificacions/classif...
4,2026-05-31,11ª Edició Marxa de l'Ordi,Ordis,Marxa Cursa,5km i 10km,https://www.curses.cat/classificacions/classif...
...,...,...,...,...,...,...
278,2017-11-11,Trail La Vall,Bellcaire d'Empordà,Trail,21km i 15km,https://www.curses.cat/classificacions/classif...
279,2017-05-20,FOXTRAIL 2017,SANT LLORENÇ DE LA MUGA,Trail,"7KM, 16KM I 27KM",https://www.curses.cat/classificacions/classif...
280,2017-01-08,La Susssissa,Espolla,Marxa Trail,10km,https://curses.cat/classificacions/classifica....
281,2016-09-30,Trail la Vall,Bellcaire d'Empordà,Trail,13km i 22km,https://www.curses.cat/classificacions/classif...


In [133]:
# Sustituimos '/' por ' i ' i ', ' por ' i '
df["Distàncies"] = df_curses["Distàncies"].str.replace("  / ", " i ", regex=False)

# Definim els teus separadors estrictes:
# 1. ', ' -> Una coma seguida exactament d'un espai
# 2. '\s+[iI]\s+' -> La lletra 'i' o 'I' separada per espais
# 3. '\n' -> El salt de línia físic que hi ha a les cel·les apilades
patron_separadores = r", |\s+[iI]\s+|\n" 

# 1. Separem la columna per aquests separadors sense perdre el text
df["distancia"] = (
    df["Distàncies"].astype(str).str.split(patron_separadores, expand=False)
)

# 2. Dupliquem les files amb 'explode' per guardar cada fragment de text en una fila nova
df_final = df.explode("distancia").reset_index(drop=True)

# 3. Neteja bàsica de control per treure espais sobrants al principi o al final de cada camp extret
df_final["distancia"] = df_final["distancia"].str.strip()

# 4. Opcional: Eliminar files buides si n'hagués quedat alguna a causa d'un salt de línia consecutiu
df_final = df_final[df_final["distancia"] != ""].reset_index(drop=True)

In [134]:
# Renombramos las columnas para que sean más descriptivas
df_final = df_final.rename(columns={
    'Cursa': 'nombre_carrera',
    'Població': 'poblacion',
    'Data': 'fecha',
    'Tipus': 'tipo_carrera'
})

In [135]:
# Reordenamos las columnas para que tengan un orden más lógico
df_final = df_final[["nombre_carrera", "poblacion", "fecha", "tipo_carrera", "Distàncies", "distancia", "URL"]]

In [136]:
df_final

,nombre_carrera,poblacion,fecha,tipo_carrera,Distàncies,distancia,URL
0,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,2026-06-21,Trail,9km i 21km,9km,https://www.curses.cat/classificacions/classif...
1,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,2026-06-21,Trail,9km i 21km,21km,https://www.curses.cat/classificacions/classif...
2,La Cirera 2026,Terrades,2026-06-07,Marxa Cursa Trail,7km i 13km,7km,https://www.curses.cat/classificacions/classif...
3,La Cirera 2026,Terrades,2026-06-07,Marxa Cursa Trail,7km i 13km,13km,https://www.curses.cat/classificacions/classif...
4,Espardenyada 2026,Sant Jaume de Llierca,2026-06-07,Marxa trail,9km i 16km,9km,https://www.curses.cat/classificacions/classif...
...,...,...,...,...,...,...,...
502,La Susssissa,Espolla,2017-01-08,Marxa Trail,10km,10km,https://curses.cat/classificacions/classifica....
503,Trail la Vall,Bellcaire d'Empordà,2016-09-30,Trail,13km i 22km,13km,https://www.curses.cat/classificacions/classif...
504,Trail la Vall,Bellcaire d'Empordà,2016-09-30,Trail,13km i 22km,22km,https://www.curses.cat/classificacions/classif...
505,Trail la Vall,Bellcaire d'Empordà,2015-11-11,Trail,20km i 15km,20km,https://www.curses.cat/classificacions/classif...


In [137]:
# 1. Asegurar que tratamos la columna como texto, en minúsculas y sin espacios
df_final["distancia"] = df_final["distancia"].astype(str).str.lower().str.strip()

# 2. ¡PRIMERO REEMPLAZAR PALABRAS CLAVE! (Antes de borrar las letras)
# Usamos un bucle con .str.replace para que actúe aunque haya más texto en la celda
reemplazos = {
    "mitja marató": "21.1",
    "mitja": "21.1",
    "marató": "42.2",
    "milla": "1.6",
}
for palabra, valor in reemplazos.items():
    df_final["distancia"] = df_final["distancia"].str.replace(palabra, valor, regex=False)

# 3. Reemplazar comas por puntos decimales
df_final["distancia"] = df_final["distancia"].str.replace(",", ".", regex=False)

# 4. AHORA SÍ: Limpiar cualquier carácter sobrante que NO sea número o punto (como "km")
df_final["distancia"] = df_final["distancia"].str.replace(r"[^\d.]", "", regex=True)

# 5. Si el valor es mayor a 100, asumimos que son metros y lo convertimos a kilómetros
# Añadimos una comprobación para evitar errores con celdas que hayan quedado vacías ("")
df_final["distancia"] = df_final["distancia"].apply(
    lambda x: str(float(x) / 1000) if x and x.replace(".", "").isdigit() and float(x) > 100 else x
)

# 6. Convertir a numérico de forma segura
df_final["distancia"] = pd.to_numeric(df_final["distancia"], errors="coerce")

In [138]:
df_final

,nombre_carrera,poblacion,fecha,tipo_carrera,Distàncies,distancia,URL
0,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,2026-06-21,Trail,9km i 21km,9.0,https://www.curses.cat/classificacions/classif...
1,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,2026-06-21,Trail,9km i 21km,21.0,https://www.curses.cat/classificacions/classif...
2,La Cirera 2026,Terrades,2026-06-07,Marxa Cursa Trail,7km i 13km,7.0,https://www.curses.cat/classificacions/classif...
3,La Cirera 2026,Terrades,2026-06-07,Marxa Cursa Trail,7km i 13km,13.0,https://www.curses.cat/classificacions/classif...
4,Espardenyada 2026,Sant Jaume de Llierca,2026-06-07,Marxa trail,9km i 16km,9.0,https://www.curses.cat/classificacions/classif...
...,...,...,...,...,...,...,...
502,La Susssissa,Espolla,2017-01-08,Marxa Trail,10km,10.0,https://curses.cat/classificacions/classifica....
503,Trail la Vall,Bellcaire d'Empordà,2016-09-30,Trail,13km i 22km,13.0,https://www.curses.cat/classificacions/classif...
504,Trail la Vall,Bellcaire d'Empordà,2016-09-30,Trail,13km i 22km,22.0,https://www.curses.cat/classificacions/classif...
505,Trail la Vall,Bellcaire d'Empordà,2015-11-11,Trail,20km i 15km,20.0,https://www.curses.cat/classificacions/classif...


In [139]:
# Descargamos el archivo final en Excel para su posterior análisis
df_final.to_excel("../../data/raw/cursescat/Curses_CAT_final.xlsx", index=False)

## Etapa 2: scraping de resultats (homes/dones/temps) per cursa

Llegeix `Curses_CAT.xlsx` (generat a l'etapa 1) i, per cada cursa, scrapeja la pàgina de classificacions per obtenir el nombre d'homes, dones i el primer temps de cada distància.

In [21]:
from pathlib import Path
import re
import time
import logging

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; curses-scraper/1.0)"}
REQUEST_TIMEOUT = 15
DELAY = 0.8
MAX_RETRIES = 3


# ---------------------------------------------------------------------------
# Helpers HTTP i parsing
# ---------------------------------------------------------------------------

def fetch(url, session):
    for intent in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            time.sleep(DELAY)
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            log.warning(f"Intent {intent}/{MAX_RETRIES} fallit: {e}")
            time.sleep(DELAY * 2)
    return None


def build_tipus_url(base_url, tipus):
    parsed = urlparse(base_url)
    qs = parse_qs(parsed.query, keep_blank_values=True)
    qs["tipus"] = [tipus]
    new_query = urlencode({k: v[0] for k, v in qs.items()})
    return urlunparse(parsed._replace(query=new_query))


def parse_select_options(soup):
    """Retorna {index: [codis]} ex: {'0': ['g0','f0','m0'], '1': ['g1','f1','m1']}"""
    groups = {}
    sel = soup.find("select")
    if not sel:
        return groups
    for opt in sel.find_all("option"):
        val = opt.get("value", "")
        m = re.search(r"tipus=([gfm])(\d+)", val)
        if m:
            genere, idx = m.group(1), m.group(2)
            groups.setdefault(idx, []).append(f"{genere}{idx}")
    return groups


def count_rows(soup):
    table = soup.find("table")
    if not table:
        return 0
    rows = table.find_all("tr")
    return max(0, len(rows) - 1)


def get_first_time(soup):
    table = soup.find("table")
    if not table:
        return None
    rows = table.find_all("tr")
    for row in rows[1:]:
        for cell in row.find_all("td"):
            text = cell.get_text(strip=True)
            if re.match(r"\d{1,2}:\d{2}:\d{2}", text):
                return text
    return None


def time_to_seconds(t):
    if not t:
        return 0
    parts = list(map(int, t.strip().split(":")))
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    return 0


def is_processable(url):
    if not isinstance(url, str):
        return False
    if "classifica.php" not in url:
        return False
    if url.endswith(".pdf") or url.endswith(".html"):
        return False
    return True


# ---------------------------------------------------------------------------
# Emparellament
# ---------------------------------------------------------------------------

def emparejar_distancies(df_url, dist_data):
    df = df_url.copy()

    mask_amb  = df["distancia"].notna()
    df_amb    = df[mask_amb]
    df_sense  = df[~mask_amb]

    n_files_total = len(df)
    n_web         = len(dist_data)

    # Si web té més categories que files Excel, descartem la de temps més curt
    categories_descartades = []
    if n_web > n_files_total:
        dist_data_sorted = sorted(dist_data, key=lambda x: x["temps_seg"])
        n_descartar = n_web - n_files_total
        categories_descartades = dist_data_sorted[:n_descartar]
        dist_data = dist_data_sorted[n_descartar:]
        log.warning(
            f"    ⚠ Web té {n_web} categories i Excel {n_files_total} files → "
            f"es descarta(en) la(es) {n_descartar} categoria(es) de temps més curt: "
            + ", ".join(f"idx={d['idx']} ({d['temps_str']})" for d in categories_descartades)
        )

    dist_data_restants = dist_data.copy()
    assignacions = {}

    # --- Pas 1: emparellar files amb distancia per proximitat normalitzada ---
    if len(df_amb) > 0 and len(dist_data_restants) > 0:
        max_km  = df_amb["distancia"].max() or 1
        max_seg = max(d["temps_seg"] for d in dist_data_restants) or 1

        df_amb_sorted = df_amb.sort_values("distancia")

        for fila_idx, fila in df_amb_sorted.iterrows():
            if not dist_data_restants:
                break
            km_norm = (fila["distancia"] or 0) / max_km
            millor  = min(
                dist_data_restants,
                key=lambda d: abs(km_norm - d["temps_seg"] / max_seg)
            )
            assignacions[fila_idx] = millor
            dist_data_restants.remove(millor)

    # --- Pas 2: files sense distancia → categories sobrants ---
    dist_data_restants_sorted = sorted(dist_data_restants, key=lambda x: x["temps_seg"])
    for fila_idx, _ in df_sense.iterrows():
        if not dist_data_restants_sorted:
            break
        assignacions[fila_idx] = dist_data_restants_sorted.pop(0)

    # --- Pas 3: escriure resultats ---
    for fila_idx, d in assignacions.items():
        total = d["homes"] + d["dones"]
        pct   = round(d["dones"] / total * 100, 1) if total > 0 else None
        df.at[fila_idx, "homes"]     = d["homes"]
        df.at[fila_idx, "dones"]     = d["dones"]
        df.at[fila_idx, "total"]     = total
        df.at[fila_idx, "pct_dones"] = pct
        df.at[fila_idx, "temps_1r"]  = d["temps_str"]
        estat = "OK"
        if categories_descartades:
            descart_info = ", ".join(f"idx={d['idx']} ({d['temps_str']})" for d in categories_descartades)
            estat = f"OK (categoria(es) descartada(es): {descart_info})"
        df.at[fila_idx, "estat"] = estat

    # Files sense assignació
    totes_assignades = set(assignacions.keys())
    for fila_idx in df.index:
        if fila_idx not in totes_assignades:
            df.at[fila_idx, "estat"] = "Sense dades web"

    return df


# ---------------------------------------------------------------------------
# Funció principal
# ---------------------------------------------------------------------------

def scrape_curses(input_excel, limit=None, desde=0):
    """
    Llegeix l'Excel (una fila per distància), fa scraping i afegeix:
      homes, dones, total, pct_dones, temps_1r, estat

    Assumpcions:
      - El Excel sempre té >= files que distàncies trobades al web per cada URL
      - Les files amb distancia s'empapellen per proximitat km↔temps normalitzat
      - Si web té més categories sobrants que files sense distancia, no s'omple cap
    """
    df = pd.read_excel(input_excel)
    df = df.dropna(subset=["URL"]).reset_index(drop=True)

    if desde:
        df = df.iloc[desde:].reset_index(drop=True)
    if limit:
        df = df.head(limit)

    # Columnes noves
    for col in ["homes", "dones", "total", "pct_dones", "temps_1r", "estat"]:
        df[col] = None

    session = requests.Session()
    urls_uniques = df["URL"].unique()
    log.info(f"Processant {len(df)} files ({len(urls_uniques)} URLs úniques)...")

    rows_out = []

    for url in urls_uniques:
        mask      = df["URL"] == url
        df_url    = df[mask].copy().reset_index(drop=True)
        nom_cursa = df_url["nombre_carrera"].iloc[0]

        if not is_processable(url):
            df_url["estat"] = "URL no processable"
            rows_out.append(df_url)
            continue

        log.info(f"  {nom_cursa}")

        soup_base = fetch(url, session)
        if not soup_base:
            df_url["estat"] = "Error carregant pàgina"
            rows_out.append(df_url)
            continue

        groups = parse_select_options(soup_base)
        if not groups:
            df_url["estat"] = "Selector no trobat"
            rows_out.append(df_url)
            continue

        # Scraping de cada índex de distància
        dist_data = []
        for idx in sorted(groups.keys(), key=int):
            tipus_codes = groups[idx]

            temps_str, temps_seg = None, 0
            for prefix in ["g", "m", "f"]:
                code = f"{prefix}{idx}"
                if code in tipus_codes:
                    s = fetch(build_tipus_url(url, code), session)
                    if s:
                        t = get_first_time(s)
                        if t:
                            temps_str = t
                            temps_seg = time_to_seconds(t)
                            break

            dones = 0
            if f"f{idx}" in tipus_codes:
                s = fetch(build_tipus_url(url, f"f{idx}"), session)
                if s:
                    dones = count_rows(s)

            homes = 0
            if f"m{idx}" in tipus_codes:
                s = fetch(build_tipus_url(url, f"m{idx}"), session)
                if s:
                    homes = count_rows(s)

            dist_data.append({
                "idx": idx, "temps_str": temps_str,
                "temps_seg": temps_seg, "homes": homes, "dones": dones,
            })
            log.info(f"    índex {idx}: 1r={temps_str}, homes={homes}, dones={dones}")

        # Emparellar files Excel amb categories web
        df_url_result = emparejar_distancies(df_url, dist_data)
        rows_out.append(df_url_result)

    df_result = pd.concat(rows_out, ignore_index=True)
    log.info(f"Fet! {len(df_result)} files generades.")
    return df_result


# ---------------------------------------------------------------------------
# Ús al Jupyter:
#
#   %run scrape_curses_v4.py
#
#   # Proves amb les primeres 10 files
#   df = scrape_curses(r"C:\...\Curses_CAT.xlsx", limit=10)
#   df
#
#   # Tot
#   df = scrape_curses(r"C:\...\Curses_CAT.xlsx")
#   df.to_excel(r"C:\...\resultats.xlsx", index=False)
# ---------------------------------------------------------------------------

In [22]:
df = scrape_curses("../../data/raw/cursescat/Curses_CAT.xlsx")
df.to_excel(r"../../data/raw/cursescat/resultats.xlsx", index=False)

21:53:22 [INFO] Processant 500 files (276 URLs úniques)...
21:53:22 [INFO]   Olla de Tapis 2026
21:53:27 [INFO]     índex 0: 1r=02:38:29, homes=55, dones=15
21:53:30 [INFO]     índex 1: 1r=00:42:40, homes=82, dones=56
21:53:30 [INFO]   La Cirera 2026
21:53:34 [INFO]     índex 0: 1r=00:30:36, homes=36, dones=58
21:53:37 [INFO]     índex 1: 1r=01:00:14, homes=160, dones=83
21:53:37 [INFO]   Espardenyada 2026
21:53:41 [INFO]     índex 0: 1r=00:57:36, homes=68, dones=70
21:53:44 [INFO]     índex 1: 1r=04:05:29, homes=125, dones=60
21:53:44 [INFO]   3ª Edició d'els Corriols de Colomers
21:53:47 [INFO]     índex 0: 1r=01:00:28, homes=86, dones=47
21:53:47 [INFO]   11ª Edició Marxa de l'Ordi
21:53:51 [INFO]     índex 0: 1r=00:39:29, homes=72, dones=90
21:53:54 [INFO]     índex 1: 1r=00:57:28, homes=89, dones=49
21:53:54 [INFO]   TerresBlanques 2026
21:53:58 [INFO]     índex 0: 1r=00:49:39, homes=63, dones=46
21:54:01 [INFO]     índex 1: 1r=01:43:20, homes=36, dones=11
21:54:01 [INFO]   Marxa 